# Lesson 2: Two bots talk

For each cell: **guess what it will print first**, then press ▶ and check.

From [Build an AI Team](https://gradlabs.in/learn/agent-team/lessons/lesson2.html).

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# The shared chat: a list of messages, in order.
# Each message is a dictionary: who wrote it, and what it says.
chat = []

def post(who, text):
    chat.append({"who": who, "text": text})
    print(f"{who}: {text}")

In [ ]:
# A bot: a name, its own instructions, and an LLM to write with
researcher = {"name": "Researcher",
              "instructions": "You are the Researcher at Amal Designers, a kids' dress brand. Find trends and prices. Say where each fact came from."}
marketer = {"name": "Marketer",
            "instructions": "You are the Marketer at Amal Designers. Write one Instagram caption from the facts in the chat. When it is final, end with DONE."}

In [ ]:
# What a bot reads: its own instructions + the chat so far.
# The last line is its own name, so the LLM writes as that bot.
def text_for(bot):
    lines = [bot["instructions"], "", "Chat so far:"]
    for m in chat:
        lines.append(m["who"] + ": " + m["text"])
    lines.append(bot["name"] + ":")
    return "\n".join(lines)

In [ ]:
# Fake LLMs: each bot's replies, written by hand, in order
researcher_replies = ["Pastel frocks are big for birthdays this month (from 3 kids' shops). Most cost ₹1,200 to ₹1,800.",
                      "Yes: buyers in those shops' reviews ask for handmade. Say that."]
marketer_replies = ["Draft: “Twirl-ready pastel frocks for birthdays 💫 DM to order!” Researcher, anything to add?",
                    "Final: “Twirl-ready pastel frocks for birthdays 💫 Handmade by Amal Designers. DM to order!” DONE"]

def fake_researcher(text):
    return researcher_replies.pop(0)

def fake_marketer(text):
    return marketer_replies.pop(0)

researcher["llm"] = fake_researcher
marketer["llm"] = fake_marketer

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# Taking turns: turn 0 → bot 0, turn 1 → bot 1, turn 2 → bot 0 …
# (% gives what's left over after dividing: 2 % 2 is 0, 3 % 2 is 1)
print(3 % 2)

def run_chat(task, bots, turn_limit=6):
    post("CEO", task)
    for turn in range(turn_limit):
        bot = bots[turn % len(bots)]
        reply = bot["llm"](text_for(bot))
        post(bot["name"], reply)
        if "DONE" in reply:
            return "Done after " + str(turn + 1) + " turns."
    return "Stopped: the turn limit was reached."

print(run_chat("Write a caption for the new pastel party frock.", [researcher, marketer]))

## Now a real LLM

The same code, with a small free model (Qwen2.5 1.5B) instead of the fake LLM.
The first run downloads it (about 3 GB), so give it a few minutes.

Small models make mistakes, and may not write DONE. The turn limit stops the chat anyway.

In [ ]:
# A real LLM: a small, free model that runs inside Colab
# (faster with a GPU: Runtime → Change runtime type → T4 GPU)
import torch                        # the library that does the model's math
from transformers import pipeline   # a ready-made way to load a model and use it

In [ ]:
# Load Qwen2.5 1.5B (about 3 GB, downloaded the first time).
# device=0 means "use the GPU", -1 means "use the CPU": it picks the GPU if there is one.
model = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
                 device=0 if torch.cuda.is_available() else -1)

NAMES = ["CEO", "Researcher", "Marketer"]

def real_llm(text):
    # The model gets one message: all the text the bot reads.
    # max_new_tokens=80: write at most 80 tokens (pieces of words).
    # do_sample=False: always pick the most likely next piece, so every run is the same.
    out = model([{"role": "user", "content": text}], max_new_tokens=80, do_sample=False)
    reply = out[0]["generated_text"][-1]["content"]   # the model's reply: the last message
    kept = []
    for line in reply.strip().split("\n"):
        if line.split(":")[0] in NAMES:   # a line that starts with a name, like "CEO:"
            if kept:
                break                     # someone else's turn: cut the reply here
            line = line.split(":", 1)[1]  # its own name again at the start: drop it
        kept.append(line.strip())
    return " ".join(kept).strip()

In [ ]:
# The Researcher has no search tool yet, so tell it not to make up facts
researcher["instructions"] = "You are the Researcher at Amal Designers, a kids' dress brand. Suggest one idea for the post in one sentence. You can't look things up, so never make up facts or prices."

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# Start a new chat, and give both bots the real LLM
chat.clear()   # empty the list
researcher["llm"] = real_llm
marketer["llm"] = real_llm
print(run_chat("Write a caption for the new pastel party frock.", [researcher, marketer]))